# ĐỒ ÁN MÔN KHAI PHÁ DỮ LIỆU (KPDL)
**Chủ đề:** Ứng dụng Thuật toán WCM-SPADE và các mô hình cơ bản (Apriori, Phân lớp, Hồi quy) vào phân tích hành vi tiêu dùng Ngân hàng & Bán lẻ.
**Nhóm:** 17

---
## PHẦN LÝ THUYẾT & DEMO CỐT LÕI TẠI COLAB
Notebook này được thiết kế để vừa chạy code vừa giải thích lý thuyết nhằm bảo vệ đồ án trước các câu hỏi vấn đáp chuyên sâu.

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

print("Đã nạp thành công thư viện chuẩn bị cho các Chương 2, 3, 5.")

## CHƯƠNG 2: TẬP PHỔ BIẾN VÀ LUẬT KẾT HỢP (Apriori vs WCM-SPADE)

### ❓ LÝ THUYẾT CHUYÊN SÂU:
- **Bài toán:** Khám phá các mối liên hệ ẩn giữa các item trong dữ liệu giao dịch.
- **Độ đo cốt lõi:**
  - **Support (Độ hỗ trợ):** Tỷ lệ giao dịch chứa tập item. *Tại sao dùng?* Lọc bỏ nhiễu, các item quá hiếm (outlier).
  - **Confidence (Độ tin cậy):** Mức độ chắc chắn luật X -> Y xảy ra.
  - **Lift (Hệ số tương quan):** *Tại sao dùng Lift khi đã có Conf?* Conf cao có thể do Y xuất hiện quá nhiều trong tập dữ liệu (ví dụ: Ai cũng mua Muối). Lift > 1 mới chứng minh X thực sự tác động thúc đẩy mua Y.

### ❓ SO SÁNH APRIORI VÀ WCM-SPADE:
1. **Khái niệm Thời gian:** Apriori xét các mặt hàng trong **CÙNG MỘT GIỎ HÀNG** (cùng thời điểm, ko xét trước sau). SPADE là mẫu tuần tự (Sequential), phân tích **CHUỖI THỜI GIAN** (A mua trước rồi mới đến B).
2. **Khái niệm Trọng số (WCM):** Apriori giả định **mọi mặt hàng có giá trị bằng nhau**. WCM-SPADE tích hợp trọng số (giá tiền) giúp phát hiện các luật tuy ít xảy ra nhưng **mang lại giá trị kinh tế khổng lồ**.

In [ ]:
# [DEMO] APRIORI ĐƠN GIẢN (KHÔNG DÙNG THƯ VIỆN ĐỂ NẮM RÕ BẢN CHẤT)
baskets = [
    ['Sữa', 'Bỉm', 'Bia'],
    ['Sữa', 'Bỉm', 'Bánh mì'],
    ['Sữa', 'Bánh mì', 'Bơ'],
    ['Bỉm', 'Bia'],
    ['Sữa', 'Bỉm', 'Bia', 'Bánh mì']
]

# 1. Tính Support (1-itemset)
n_trans = len(baskets)
min_sup = 0.4 # 40%
item_counts = defaultdict(int)
for b in baskets:
    for item in b: item_counts[item] += 1

freq_1 = {k: v/n_trans for k, v in item_counts.items() if (v/n_trans) >= min_sup}
print("Tập phổ biến 1-item (Support >= 40%):\n", freq_1)

## CHƯƠNG 3: PHÂN LỚP DỮ LIỆU (Classification)

### ❓ LÝ THUYẾT CHUYÊN SÂU:
- **Tại sao dùng Phân lớp ở đây?** Để phân loại khách hàng VIP (Chi tiêu cao) và Normal (Chi tiêu thấp) dựa trên hành vi (Tần suất, Loại mặt hàng).
- **So sánh 3 mô hình đã dùng:**
  1. **Decision Tree (Cây quyết định):** 
     - *Điểm mạnh:* White-box, dễ trích xuất luật If-Then để giải thích cho con người.
     - *Điểm yếu:* Overfitting nếu không cắt tỉa (pruning).
  2. **Naive Bayes:**
     - *Điểm mạnh:* Nhanh, nhẹ, xử lý tốt nhiễu (Dựa trên định lý Bayes).
     - *Điểm yếu:* "Naive" - Giả định ngây thơ rằng các thuộc tính độc lập nhau. (Thực tế: Số lần mua và Tổng tiền mua cực kỳ liên quan).
  3. **KNN (K-Nearest Neighbors):**
     - *Điểm mạnh:* Thuật toán lười (Lazy learning), phi tham số. Gán nhãn dựa trên K khách hàng gần giống nhất.
     - *Điểm yếu:* **Tại sao bắt buộc phải dùng StandardScaler?** KNN dùng khoảng cách Euclidean. Cột "Tổng tiền" (vài triệu) sẽ áp đảo cột "Tần suất" (vài chục) nếu không chuẩn hóa về cùng tỷ lệ.

In [ ]:
# [DEMO] PHÂN LỚP VỚI 3 MÔ HÌNH VÀ CHUẨN HÓA KẾT QUẢ
np.random.seed(42)
# Features: [Tần suất, Tổng ngày active, Số loại mặt hàng mua, Trung bình chi]
X = np.random.rand(100, 4) * [50, 100, 20, 1000]
y = (X[:, 3] > 600).astype(int) # Nhãn: 1 nếu chi tiêu trung bình > 600 (VIP)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- KNN bắt buộc phải có StandardScaler ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    "Decision Tree": (DecisionTreeClassifier(max_depth=4), False),
    "Naive Bayes": (GaussianNB(), True),
    "KNN (k=5)": (KNeighborsClassifier(n_neighbors=5), True)
}

for name, (clf, use_scaled) in models.items():
    X_tr = X_train_s if use_scaled else X_train
    X_te = X_test_s if use_scaled else X_test
    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    print(f"{name:15s} | Accuracy: {accuracy_score(y_test, preds):.2f}")

## CHƯƠNG 5: HỒI QUY VÀ DỰ BÁO (Regression)

### ❓ LÝ THUYẾT CHUYÊN SÂU:
- **Tại sao vừa dùng Phân Lớp vừa dùng Hồi Quy?**
  - Phân lớp: Dự đoán biến rời rạc (VIP hoặc Thường).
  - Hồi quy: Dự đoán biến liên tục (Chính xác tháng tới chi tiêu bao nhiêu VND).
- **Ý nghĩa của Hồi quy tuyến tính (Linear Regression):** Phương trình $Y = a_1X_1 + a_2X_2 + b$. 
  - Các hệ số $a$ (Coefficients) tìm được cho biết sự ảnh hưởng của các tháng trước lên tháng tới.
- **Sự khác biệt giữa độ đo MAE và R-squared ($R^2$):**
  - **MAE (Mean Absolute Error):** Đo độ lệch trung bình. Có đơn vị (VND). Giúp người kinh doanh hiểu "Model này dự đoán sai bao nhiêu tiền".
  - **R-squared:** Không có đơn vị. Đo mức độ phù hợp của mô hình (từ 0 đến 1). Mức độ biến đổi của Y được X giải thích là bao nhiêu %.

In [ ]:
# [DEMO] HỒI QUY TUYẾN TÍNH
# X: Chi tiêu [Tháng 1, Tháng 2, Tháng 3]
# y: Chi tiêu Tháng 4
X_reg = np.array([
    [100, 110, 105],
    [200, 180, 210],
    [50,  60,  55],
    [300, 310, 350]
])
y_reg = np.array([115, 205, 65, 340])

reg = LinearRegression()
reg.fit(X_reg, y_reg)
y_pred_reg = reg.predict(X_reg)

print("Hệ số phương trình (Coefficients):", reg.coef_)
print("Hệ số chặn (Intercept):", reg.intercept_)
print("MAE (Trung bình sai lệch):", mean_absolute_error(y_reg, y_pred_reg))